# 2. 首次构建 COF（Windows 版）

语言：[English](../en/02_first_cof_build.ipynb) | **中文**

本 Notebook 使用共享的 `cofkit` 环境，在 `hcb` 拓扑上构建 TAPB–TFB 亚胺 COF。可运行单元格通过 Python 的 `subprocess` 模块传递 cofkit CLI 参数；其下方的直接 Python API 工作流仍保持注释状态。

## 教程系列

1. [环境设置](01_environment_setup.ipynb)
2. **首次构建 COF**（本 Notebook）
3. [拓扑、连接数组合与连接键示例](03_topologies_connectivities_and_linkages.ipynb)
4. [使用 Zeo++ 进行孔隙分析](04_zeopp_pore_analysis.ipynb)

请先运行 [Notebook 1](01_environment_setup.ipynb)。本 Notebook 将 `Python (cofkit)` 声明为默认内核，同时也显式使用 `conda run -n cofkit`。

## 在 `hcb` 拓扑上构建 TAPB–TFB

TAPB 和 TFB 各自具有三个反应基团，因此形成 3+3 连接数组合。`imine_bridge` 模板连接胺基和醛基，而 `--topology hcb` 则让拓扑选择明确且可复现。

该命令会写入 `summary.json`，并把导出的 CIF 按粗略验证类别存放在 `out/tutorial_02_first_build/cifs/` 下。

In [ ]:
import os
from pathlib import Path
import shutil
import subprocess

repo_root = Path.cwd().resolve()
conda = os.environ.get("CONDA_EXE") or shutil.which("conda")
if not conda:
    raise RuntimeError("未找到 Conda。请先运行环境设置单元格。")

command = [
    conda, "run", "-n", "cofkit", "cofkit",
    "build", "single-pair",
    "--template-id", "imine_bridge",
    "--first-smiles", "C1=CC(=CC=C1C2=CC(=CC(=C2)C3=CC=C(C=C3)N)C4=CC=C(C=C4)N)N",
    "--second-smiles", "C1=C(C=C(C=C1C=O)C=O)C=O",
    "--first-id", "tapb",
    "--second-id", "tfb",
    "--first-motif-kind", "amine",
    "--second-motif-kind", "aldehyde",
    "--topology", "hcb",
    "--target-dimensionality", "2D",
    "--output-dir", "out/tutorial_02_first_build",
]
subprocess.run(command, cwd=repo_root, check=True)

In [ ]:
# Python API 等价实现（已注释）。
# from pathlib import Path
# from cofkit import BatchGenerationConfig, BatchStructureGenerator, build_rdkit_monomer
#
# tapb = build_rdkit_monomer(
#     "tapb",
#     "TAPB",
#     "C1=CC(=CC=C1C2=CC(=CC(=C2)C3=CC=C(C=C3)N)C4=CC=C(C=C4)N)N",
#     "amine",
#     num_conformers=4,
# )
# tfb = build_rdkit_monomer(
#     "tfb",
#     "TFB",
#     "C1=C(C=C(C=C1C=O)C=O)C=O",
#     "aldehyde",
#     num_conformers=4,
# )
# generator = BatchStructureGenerator(
#     BatchGenerationConfig(
#         allowed_reactions=("imine_bridge",),
#         target_dimensionality="2D",
#         topology_ids=("hcb",),
#         rdkit_num_conformers=4,
#         max_workers=1,
#         write_cif=True,
#     )
# )
# summaries, candidates, attempted = generator.generate_monomer_pair_candidates(
#     tapb,
#     tfb,
#     pair_id="tapb__tfb",
#     out_dir=Path("out/tutorial_02_first_build_python/cifs"),
# )
# print("attempted:", attempted)
# for summary in summaries:
#     print(summary.topology_id, summary.status, summary.score, summary.cif_path)

## 查看构建输出

CIF 可能出现在 `valid`、`warning` 或 `needs_optimization` 目录中；这些目录表示粗略的几何分类，而不是不同的化学体系。Notebook 4 会搜索所有这些子目录，不会预设某一个类别。

In [ ]:
import json
from pathlib import Path

output_dir = Path("out/tutorial_02_first_build")
summary_path = output_dir / "summary.json"
if not summary_path.is_file():
    raise FileNotFoundError("请先运行首次构建单元格，再查看输出。")

summary = json.loads(summary_path.read_text(encoding="utf-8"))
print("--- summary.json ---")
print(json.dumps(summary, indent=2))
print("\n--- 导出的 CIF 文件 ---")
print(*(path.resolve() for path in sorted((output_dir / "cifs").rglob("*.cif"))), sep="\n")

In [ ]:
# Python 等价实现（已注释）：查看结构化报告并查找 CIF。
# import json
# from pathlib import Path
# output_dir = Path("out/tutorial_02_first_build")
# summary = json.loads((output_dir / "summary.json").read_text())
# print(summary["successful_structures"], summary["cifs_written"])
# print(*sorted((output_dir / "cifs").rglob("*.cif")), sep="\n")

## 需要保留和关注的内容

- `summary.json` 记录了所请求的化学体系、检测到的基元数量、尝试的结构、评分、验证标记和 CIF 路径。
- 每个 CIF 都以 `# COFid:` 注释开头，其中包含单体、拓扑和连接键。
- 这些是初始生成结构。标记为警告或 `needs_optimization` 的结果，应在模拟或发表前完成优化并重新验证。
- 请保留 `out/tutorial_02_first_build/`：Notebook 4 会把其中的 CIF 作为默认 Zeo++ 输入。

继续学习 [Notebook 3](03_topologies_connectivities_and_linkages.ipynb)，比较少量具有代表性的其他构建选择。